# EuroSAT embedding anisotropy

This notebook uses the official GEO-Bench `m-eurosat` splits to compare pre-norm and post-norm encoder embeddings. It measures effective rank, covariance concentration, and pairwise cosine before visualizing the same validation samples with t-SNE.

## TL;DR

Run the notebook before filling in a conclusion. Higher effective rank and lower top-PC variance indicate a less concentrated covariance spectrum. A lower raw cosine alone is not sufficient because it can be caused by removing a shared mean. t-SNE is descriptive and must not replace a frozen linear probe.

## Context & Methods

For centered embeddings with covariance eigenvalues $\lambda_i$, define $p_i=\lambda_i/\sum_j\lambda_j$. The effective rank is

$$r_{\mathrm{eff}}=\exp\left(-\sum_i p_i\log p_i\right).$$

It is a smooth measure of how many independent directions carry substantial variance; it is not the number of exactly nonzero dimensions. Pairwise cosine measures angular alignment between samples. The mean-direction ratio $\rho_\mu=\|\mathbb{E}[x]\|/\mathbb{E}[\|x\|]$ lies between zero and one; a large value indicates a strong common direction. Neighbor hubness measures how unevenly samples occur in other samples' nearest-neighbor lists.

### Key Assumptions

- The model is frozen and routing is deterministic.
- Pre-norm and post-norm embeddings come from the same forward pass and use mean-fine pooling.
- Centering and standardization statistics are fitted only on the official training split.
- Geometry is measured on the official validation split.
- The same stratified validation subset and t-SNE seed are used for both token sources.

## 1. Parameters

Set `train_limit` and `valid_limit` to `None` for complete splits. The defaults are sufficient for a stable diagnostic while keeping extraction bounded.

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

run_dir = Path(os.environ.get('MEOX_RUN_DIR', repo_root))
config_path = Path(os.environ.get(
    'MEOX_CONFIG', repo_root / 'configs/pretrain_mmearth_moe_mae_full.yaml'
))
checkpoint_path = Path(os.environ.get(
    'MEOX_CHECKPOINT', repo_root / 'weights/pretrained/meox_s_mmearth64_best.pth'
))
geobench_root = Path(os.environ['GEO_BENCH_DIR'])
cache_dir = run_dir / 'analysis/geobench_eurosat_anisotropy'

dataset_name = 'm-eurosat'
partition_name = 'default'
pooling = 'mean_fine'
train_limit = 5000
valid_limit = 5000
batch_size = 128
num_workers = 4
sample_seed = 42
device_override = None
reuse_cache = True

hubness_k = 10
hubness_max_samples = 3000

tsne_transform = 'standardized'  # 'raw', 'centered', or 'standardized'
max_tsne_samples = 1000
tsne_pca_dimensions = 30
tsne_max_iter = 500
tsne_threads = 4
tsne_seed = 42

pc_removal_values = (0, 1, 2, 4, 8, 16, 32)
retrieval_k = 5
probe_max_iter = 1000

## 2. Load EuroSAT and the frozen encoder

In [ ]:
import sys
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE, trustworthiness
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neighbors import NearestNeighbors
from scipy.stats import skew
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

sys.path.insert(0, str(repo_root))

from datasets.geobench import GeoBenchClassificationDataset
from utils.extract_embeddings import (
    build_model_from_config,
    load_config,
    make_inference_dataloader,
    maybe_limit_dataset,
    model_input_schema,
    move_to_device,
    resolve_device,
)

In [ ]:
assert config_path.exists(), f'Missing config: {config_path}'
assert checkpoint_path.exists(), f'Missing checkpoint: {checkpoint_path}'
assert (geobench_root / 'classification_v1.0' / dataset_name).exists(), (
    f'Missing GEO-Bench EuroSAT data under {geobench_root}'
)

config = load_config(str(config_path))
device = resolve_device(device_override)
_, model_band_names, _ = model_input_schema(config)

train_base = GeoBenchClassificationDataset(
    root_dir=str(geobench_root), dataset_name=dataset_name, split='train',
    model_band_names=model_band_names, partition_name=partition_name,
)
valid_base = GeoBenchClassificationDataset(
    root_dir=str(geobench_root), dataset_name=dataset_name, split='valid',
    model_band_names=model_band_names, partition_name=partition_name,
)
train_dataset = maybe_limit_dataset(train_base, train_limit, sample_seed)
valid_dataset = maybe_limit_dataset(valid_base, valid_limit, sample_seed)
model = build_model_from_config(config, str(checkpoint_path), device)

print(f'device={device} train={len(train_dataset)} valid={len(valid_dataset)}')
print('input bands:', valid_base.raster_band_names)
print('source mapping:', valid_base.band_mapping)
print('preprocessing:', valid_base.preprocessing_signature)
print('checkpoint:', checkpoint_path)

## 3. Extract pre-norm and post-norm embeddings

Both representations are pooled from one encoder output. This avoids duplicate inference and guarantees that routing and input batches are identical. Cached files contain embeddings, labels, and sample IDs only.

In [ ]:
@torch.inference_mode()
def extract_pre_post(active_model, dataset, dataset_info):
    dataloader = make_inference_dataloader(dataset, batch_size, num_workers)
    collected = {'pre_norm': [], 'post_norm': [], 'labels': [], 'sample_ids': []}
    active_model.eval()
    use_amp = device.type == 'cuda'

    for batch in tqdm(dataloader, desc='Extracting pre/post embeddings'):
        rasters = move_to_device(batch['raster_dict'], device)
        validity = move_to_device(batch.get('raster_valid_masks'), device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            features = active_model.forward_features(
                raster_dict=rasters,
                raster_valid_masks=validity,
                raster_band_names=dataset_info.raster_band_names,
                return_routing=False,
                stochastic_routing=False,
            )
            for token_source in ('pre_norm', 'post_norm'):
                embedding = active_model.encoder._pool_feature_tokens(
                    features, token_source=token_source, pooling=pooling
                )
                collected[token_source].append(embedding.float().cpu())
        collected['labels'].append(batch['label'].cpu())
        collected['sample_ids'].extend(str(value) for value in batch['sample_id'])

    return {
        'pre_norm': torch.cat(collected['pre_norm']).numpy(),
        'post_norm': torch.cat(collected['post_norm']).numpy(),
        'labels': torch.cat(collected['labels']).numpy(),
        'sample_ids': np.asarray(collected['sample_ids'], dtype=str),
    }


def load_or_extract(split_name, dataset, dataset_info):
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / (
        f'{dataset_name}_{split_name}_{dataset_info.preprocessing_signature}_'
        f'{checkpoint_path.stem}_{pooling}.npz'
    )
    if reuse_cache and cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as archive:
            outputs = {name: archive[name] for name in archive.files}
        print('loaded:', cache_path)
        return outputs
    outputs = extract_pre_post(model, dataset, dataset_info)
    np.savez_compressed(cache_path, **outputs)
    print('saved:', cache_path)
    return outputs

In [ ]:
train_outputs = load_or_extract('train', train_dataset, train_base)
valid_outputs = load_or_extract('valid', valid_dataset, valid_base)

for split_name, outputs in [('train', train_outputs), ('valid', valid_outputs)]:
    print(
        split_name, outputs['pre_norm'].shape, outputs['post_norm'].shape,
        'finite=', np.isfinite(outputs['pre_norm']).all() and np.isfinite(outputs['post_norm']).all(),
    )

## 4. Geometry metrics

The covariance calculation always centers the validation embeddings, so the effective rank of `raw` and `centered` should match. Their pairwise cosine can differ strongly. Standardization additionally changes the covariance spectrum by scaling each dimension with training-set statistics.

In [ ]:
def mean_pairwise_cosine(values):
    normalized = values / np.clip(np.linalg.norm(values, axis=1, keepdims=True), 1e-12, None)
    sample_count = len(normalized)
    return float(
        (np.square(normalized.sum(axis=0)).sum() - sample_count)
        / (sample_count * (sample_count - 1))
    )


def covariance_spectrum(values):
    centered = values.astype(np.float64) - values.mean(axis=0, keepdims=True)
    covariance = centered.T @ centered / max(len(centered) - 1, 1)
    eigenvalues = np.clip(np.linalg.eigvalsh(covariance), 0, None)[::-1]
    return eigenvalues / np.clip(eigenvalues.sum(), 1e-12, None)


def geometry_metrics(values):
    spectrum = covariance_spectrum(values)
    positive = spectrum > 0
    effective_rank = np.exp(-(spectrum[positive] * np.log(spectrum[positive])).sum())
    participation_ratio = 1.0 / np.clip(np.square(spectrum).sum(), 1e-12, None)
    centered = values - values.mean(axis=0, keepdims=True)
    embedding_norms = np.linalg.norm(values, axis=1)
    residual_norm = np.linalg.norm(centered, axis=1).mean()
    return {
        'effective_rank': effective_rank,
        'participation_ratio': participation_ratio,
        'top_pc_fraction': spectrum[0],
        'top_10_pc_fraction': spectrum[:10].sum(),
        'pairwise_cosine': mean_pairwise_cosine(values),
        'mean_direction_ratio': np.linalg.norm(values.mean(axis=0)) / max(embedding_norms.mean(), 1e-12),
        'mean_to_residual_norm': np.linalg.norm(values.mean(axis=0)) / max(residual_norm, 1e-12),
        'near_constant_dimensions': int((values.std(axis=0) < 1e-6).sum()),
    }


train_statistics = {}
representations = {}
for token_source in ('pre_norm', 'post_norm'):
    train_values = train_outputs[token_source].astype(np.float64)
    valid_values = valid_outputs[token_source].astype(np.float64)
    train_mean = train_values.mean(axis=0, keepdims=True)
    train_std = train_values.std(axis=0, ddof=1, keepdims=True)
    train_statistics[token_source] = {'mean': train_mean, 'std': train_std}
    representations[(token_source, 'raw')] = valid_values
    representations[(token_source, 'centered')] = valid_values - train_mean
    representations[(token_source, 'standardized')] = (
        (valid_values - train_mean) / np.clip(train_std, 1e-8, None)
    )

geometry_table = pd.DataFrame([
    {'token_source': source, 'transform': transform, **geometry_metrics(values)}
    for (source, transform), values in representations.items()
]).set_index(['token_source', 'transform'])
display(geometry_table.round(4))

### Interpretation

- Compare effective rank and top-PC fraction between pre-norm and post-norm.
- Compare the mean-direction ratio and `raw` versus `centered` cosine to determine whether a shared mean explains angular concentration.
- Compare `standardized` rows to measure what a train-fitted downstream preprocessing step can recover.
- Do not select an embedding solely because it has higher effective rank; confirm with the frozen EuroSAT linear probe.

## 5. Covariance spectrum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for axis, transform in zip(axes, ('raw', 'standardized')):
    for token_source, color in [('pre_norm', '#176B87'), ('post_norm', '#C84B31')]:
        spectrum = covariance_spectrum(representations[(token_source, transform)])
        axis.plot(np.arange(1, 31), spectrum[:30], marker='o', markersize=3, label=token_source, color=color)
    axis.set(
        title=f'{transform.capitalize()} validation embeddings',
        xlabel='Principal component', ylabel='Fraction of variance',
    )
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()

## 6. Neighbor hubness

For each representation, build cosine $k$-nearest-neighbor lists and count how often each sample occurs in another sample's list. The mean count is always $k$, so skewness, Gini coefficient, upper-tail ratios, and the fraction never selected reveal concentration. Lower values generally mean less hubness, but genuine class structure can also create popular neighbors.

In [ ]:
def gini_coefficient(values):
    sorted_values = np.sort(np.asarray(values, dtype=np.float64))
    if sorted_values.sum() == 0:
        return 0.0
    ranks = np.arange(1, len(sorted_values) + 1)
    return float(
        2 * np.sum(ranks * sorted_values) / (len(sorted_values) * sorted_values.sum())
        - (len(sorted_values) + 1) / len(sorted_values)
    )


def neighbor_occurrences(values, k):
    neighbors = NearestNeighbors(
        n_neighbors=k + 1, metric='cosine', algorithm='brute', n_jobs=-1
    ).fit(values)
    candidate_indices = neighbors.kneighbors(values, return_distance=False)
    selected = np.empty((len(values), k), dtype=np.int64)
    for sample_index, row in enumerate(candidate_indices):
        selected[sample_index] = row[row != sample_index][:k]
    return np.bincount(selected.reshape(-1), minlength=len(values))


rng = np.random.default_rng(sample_seed)
hubness_sample_count = min(hubness_max_samples, len(valid_outputs['labels']))
hubness_indices = rng.choice(
    len(valid_outputs['labels']), size=hubness_sample_count, replace=False
)
hubness_rows = []
hubness_occurrences = {}
for token_source in ('pre_norm', 'post_norm'):
    for transform in ('raw', 'centered', 'standardized'):
        key = (token_source, transform)
        occurrences = neighbor_occurrences(representations[key][hubness_indices], hubness_k)
        hubness_occurrences[key] = occurrences
        mean_occurrence = occurrences.mean()
        hubness_rows.append({
            'token_source': token_source,
            'transform': transform,
            'k': hubness_k,
            'occurrence_skewness': skew(occurrences, bias=False),
            'occurrence_gini': gini_coefficient(occurrences),
            'zero_occurrence_fraction': np.mean(occurrences == 0),
            'p99_over_mean': np.percentile(occurrences, 99) / mean_occurrence,
            'max_over_mean': occurrences.max() / mean_occurrence,
        })

hubness_table = pd.DataFrame(hubness_rows).set_index(['token_source', 'transform'])
display(hubness_table.round(4))

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=False, sharey=False)
for row_index, token_source in enumerate(('pre_norm', 'post_norm')):
    for column_index, transform in enumerate(('raw', 'centered', 'standardized')):
        axis = axes[row_index, column_index]
        axis.hist(hubness_occurrences[(token_source, transform)], bins=30, color='#176B87', alpha=0.85)
        axis.set_title(f'{token_source} / {transform}')
        axis.set_xlabel(f'occurrences in {hubness_k}-NN lists')
        axis.set_ylabel('samples')
plt.tight_layout()
plt.show()

## 7. Matched t-SNE visualization

The same class-stratified samples are used in both panels. PCA reduces each representation to at most 50 dimensions before t-SNE. `trustworthiness` checks whether local high-dimensional neighborhoods are preserved, but neither this score nor visual class separation replaces a linear probe.

In [ ]:
labels = valid_outputs['labels'].astype(int).reshape(-1)
if len(labels) > max_tsne_samples:
    splitter = StratifiedShuffleSplit(
        n_splits=1, train_size=max_tsne_samples, random_state=tsne_seed
    )
    selected_indices, _ = next(splitter.split(np.zeros(len(labels)), labels))
else:
    selected_indices = np.arange(len(labels))
selected_labels = labels[selected_indices]

projections = {}
quality_rows = []
for token_source in ('pre_norm', 'post_norm'):
    values = representations[(token_source, tsne_transform)][selected_indices]
    pca_dimensions = min(tsne_pca_dimensions, values.shape[1], len(values) - 1)
    reduced = PCA(
        n_components=pca_dimensions, svd_solver='randomized', random_state=tsne_seed
    ).fit_transform(values)
    perplexity = min(30, max(5, (len(reduced) - 1) // 3))
    started = perf_counter()
    print(f'Starting {token_source}: n={len(reduced)}, d={pca_dimensions}, perplexity={perplexity}')
    with threadpool_limits(limits=tsne_threads):
        projection = TSNE(
            n_components=2, perplexity=perplexity, init='pca',
            learning_rate='auto', random_state=tsne_seed,
            method='barnes_hut', max_iter=tsne_max_iter, angle=0.8,
            n_jobs=tsne_threads, verbose=1,
        ).fit_transform(reduced)
    elapsed = perf_counter() - started
    print(f'Finished {token_source} in {elapsed:.1f} seconds')
    projections[token_source] = projection
    quality_rows.append({
        'token_source': token_source,
        'transform': tsne_transform,
        'samples': len(reduced),
        'elapsed_seconds': elapsed,
        'trustworthiness_k15': trustworthiness(reduced, projection, n_neighbors=15),
    })

display(pd.DataFrame(quality_rows).set_index('token_source').round(4))

In [ ]:
classes = np.unique(selected_labels)
cmap = plt.get_cmap('tab10', len(classes))
class_to_color = {class_id: cmap(index) for index, class_id in enumerate(classes)}
point_colors = [class_to_color[class_id] for class_id in selected_labels]

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), sharex=False, sharey=False)
for axis, token_source in zip(axes, ('pre_norm', 'post_norm')):
    projection = projections[token_source]
    axis.scatter(projection[:, 0], projection[:, 1], c=point_colors, s=8, alpha=0.65, linewidths=0)
    axis.set_title(f'{token_source} / {tsne_transform}')
    axis.set_xticks([])
    axis.set_yticks([])

legend = [
    Line2D([0], [0], marker='o', linestyle='', color=class_to_color[class_id], label=f'class {class_id}')
    for class_id in classes
]
fig.legend(handles=legend, loc='center right', title='EuroSAT label')
fig.suptitle(f'Matched GEO-Bench EuroSAT t-SNE, n={len(selected_indices)}', y=1.02)
plt.tight_layout(rect=(0, 0, 0.9, 1))
plt.show()

## 8. Does removing dominant PCs preserve semantic signal?

Fit PCA only on centered training embeddings. For each $k$, subtract the projection onto the first $k$ principal components and L2-normalize. Compare this sweep with train-standardized embeddings using two semantic validation metrics:

- Cosine retrieval from validation queries into the official training split: nearest-neighbor accuracy and precision@$K$.
- An unregularized multinomial logistic-regression probe. Removing regularization avoids favoring one representation merely because it has a different scale.

Choose any $k$ using validation only. The official test split should be evaluated once after fixing the representation.

In [ ]:
def l2_normalize(values):
    return values / np.clip(np.linalg.norm(values, axis=1, keepdims=True), 1e-12, None)


def retrieval_metrics(train_values, train_labels, query_values, query_labels, k):
    search = NearestNeighbors(
        n_neighbors=k, metric='cosine', algorithm='brute', n_jobs=-1
    ).fit(train_values)
    neighbor_indices = search.kneighbors(query_values, return_distance=False)
    neighbor_labels = train_labels[neighbor_indices]
    return {
        'retrieval_top1_accuracy': np.mean(neighbor_labels[:, 0] == query_labels),
        f'retrieval_precision_at_{k}': np.mean(neighbor_labels == query_labels[:, None]),
    }


def probe_accuracy(train_values, train_labels, valid_values, valid_labels):
    probe = LogisticRegression(
        penalty=None, solver='lbfgs', max_iter=probe_max_iter, random_state=sample_seed
    )
    probe.fit(train_values, train_labels)
    return probe.score(valid_values, valid_labels)


train_labels = train_outputs['labels'].astype(int).reshape(-1)
valid_labels = valid_outputs['labels'].astype(int).reshape(-1)
semantic_rows = []
for token_source in ('pre_norm', 'post_norm'):
    train_raw = train_outputs[token_source].astype(np.float64)
    valid_raw = valid_outputs[token_source].astype(np.float64)
    train_mean = train_statistics[token_source]['mean']
    train_std = train_statistics[token_source]['std']
    train_centered = train_raw - train_mean
    valid_centered = valid_raw - train_mean

    max_components = max(pc_removal_values)
    if max_components >= train_centered.shape[1]:
        raise ValueError('PC removal must retain at least one embedding dimension')
    components = PCA(n_components=max_components, svd_solver='full').fit(train_centered).components_

    for removed_components in pc_removal_values:
        active_components = components[:removed_components]
        train_filtered = train_centered - (train_centered @ active_components.T) @ active_components
        valid_filtered = valid_centered - (valid_centered @ active_components.T) @ active_components
        train_filtered = l2_normalize(train_filtered)
        valid_filtered = l2_normalize(valid_filtered)
        retrieval = retrieval_metrics(
            train_filtered, train_labels, valid_filtered, valid_labels, retrieval_k
        )
        semantic_rows.append({
            'token_source': token_source,
            'method': 'center_remove_pc_l2',
            'removed_pcs': removed_components,
            **retrieval,
            'probe_accuracy': probe_accuracy(
                train_filtered, train_labels, valid_filtered, valid_labels
            ),
        })

    train_standardized = (train_raw - train_mean) / np.clip(train_std, 1e-8, None)
    valid_standardized = (valid_raw - train_mean) / np.clip(train_std, 1e-8, None)
    retrieval = retrieval_metrics(
        train_standardized, train_labels, valid_standardized, valid_labels, retrieval_k
    )
    semantic_rows.append({
        'token_source': token_source,
        'method': 'standardized',
        'removed_pcs': np.nan,
        **retrieval,
        'probe_accuracy': probe_accuracy(
            train_standardized, train_labels, valid_standardized, valid_labels
        ),
    })

semantic_table = pd.DataFrame(semantic_rows)
display(semantic_table.round(4))

In [ ]:
metric_names = [
    'retrieval_top1_accuracy', f'retrieval_precision_at_{retrieval_k}', 'probe_accuracy'
]
metric_titles = ['Retrieval top-1 accuracy', f'Retrieval precision@{retrieval_k}', 'Linear-probe accuracy']
colors = {'pre_norm': '#176B87', 'post_norm': '#C84B31'}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, metric, title in zip(axes, metric_names, metric_titles):
    for token_source in ('pre_norm', 'post_norm'):
        sweep = semantic_table.query(
            "token_source == @token_source and method == 'center_remove_pc_l2'"
        ).sort_values('removed_pcs')
        baseline = semantic_table.query(
            "token_source == @token_source and method == 'standardized'"
        )[metric].iloc[0]
        axis.plot(
            sweep['removed_pcs'], sweep[metric], marker='o', color=colors[token_source],
            label=f'{token_source}: remove PCs',
        )
        axis.axhline(
            baseline, color=colors[token_source], linestyle='--', alpha=0.7,
            label=f'{token_source}: standardized',
        )
    axis.set(title=title, xlabel='Removed leading PCs', ylabel='score')
    axis.set_xticks(pc_removal_values)
    axis.grid(alpha=0.25)
axes[-1].legend(bbox_to_anchor=(1.03, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 9. Takeaways

After execution, record the following rather than relying on the plots alone:

1. Which token source has higher centered effective rank and lower top-PC variance?
2. How much does train-set centering reduce pairwise cosine?
3. Does train-set standardization materially improve the covariance spectrum?
4. Does centering or standardization reduce neighbor-occurrence skewness and upper-tail concentration?
5. Are t-SNE structures and trustworthiness consistent between token sources?
6. Does removing the first few PCs improve retrieval and probe accuracy, indicating nuisance-dominated directions, or reduce accuracy, indicating useful signal?
7. Does the representation with better geometry also win the frozen EuroSAT linear probe?

A visually cleaner t-SNE is not evidence of a better representation. The final representation choice should be made using the official validation split and frozen-probe performance.